# Clase 046 — Web scraping con BeautifulSoup

**Parte 0** · Mitchell caps. 1-3.

> 🎯 Extraer datos cuando no hay API. Con respeto: robots.txt, delays, ética.

> ⏱️ ~75 min

## ⚙️ Setup

```bash
pip install beautifulsoup4 lxml
```

In [ ]:
import requests
import time
from bs4 import BeautifulSoup
print(f'bs4 OK')

## 1️⃣ Parsear HTML

```python
soup = BeautifulSoup(html_string, 'html.parser')
# alternativas: 'lxml' (más rápido), 'html5lib' (más permisivo)
```

In [ ]:
html = '''
<html><body>
  <div class="product" data-id="1">
    <h2 class="name">Libro Python</h2>
    <span class="price">$30</span>
    <a href="/libros/python">ver</a>
  </div>
  <div class="product" data-id="2">
    <h2 class="name">Guitarra</h2>
    <span class="price">$800</span>
    <a href="/musica/guitarra">ver</a>
  </div>
  <div class="product" data-id="3">
    <h2 class="name">Auriculares</h2>
    <span class="price">$120</span>
    <a href="/audio/auriculares">ver</a>
  </div>
</body></html>
'''

soup = BeautifulSoup(html, 'html.parser')
for prod in soup.find_all('div', class_='product'):
    nombre = prod.find('h2', class_='name').get_text(strip=True)
    precio = prod.find('span', class_='price').get_text(strip=True)
    url    = prod.find('a')['href']
    id_    = prod['data-id']
    print(f'id={id_}  {nombre:20s}  {precio:6s}  {url}')

## 2️⃣ Selectores CSS — `soup.select(...)`

Más potentes que `find_all` para queries complejos:

```python
soup.select('.product')                    # class
soup.select('#main')                        # id
soup.select('div > a')                      # hijo directo
soup.select('div.product .price')          # descendiente con class
soup.select('a[href^="/"]')                 # atributo con prefijo
soup.select('div:nth-of-type(2)')          # pseudo-class
```

In [ ]:
for precio in soup.select('.product .price'):
    print(precio.get_text(strip=True))

## 3️⃣ Páginas dinámicas (JS) — limitación

`requests` descarga el HTML **original** del servidor, sin ejecutar JavaScript. Si el contenido aparece tras AJAX/SPA frameworks (React, Vue), no lo verás.

**Soluciones**:
- `playwright` o `selenium`: navegadores headless que renderizan JS.
- Inspeccionar Network tab del DevTools — muchas veces la SPA hace XHR a un endpoint JSON que puedes consumir directo con requests (mejor que scraping).

## 4️⃣ Tabla HTML → DataFrame

pandas tiene un shortcut para tablas: `pd.read_html(url)` devuelve lista de DataFrames, uno por `<table>` en la página.

In [ ]:
import pandas as pd
html_tabla = '''
<table>
  <tr><th>producto</th><th>precio</th></tr>
  <tr><td>A</td><td>100</td></tr>
  <tr><td>B</td><td>200</td></tr>
  <tr><td>C</td><td>150</td></tr>
</table>
'''
tablas = pd.read_html(html_tabla)
print(tablas[0])

## 5️⃣ Ética y `robots.txt`

**`robots.txt`** vive en la raíz del dominio (`https://sitio.com/robots.txt`). Indica qué paths puede crawlear cada bot:

```
User-agent: *
Disallow: /private/
Disallow: /admin/
Allow: /public/
Crawl-delay: 1
```

**Reglas mínimas**:
1. Lee `robots.txt` antes de scrapear (`urllib.robotparser` lo parsea).
2. **Respeta `Crawl-delay`** y, si no existe, pon mínimo 1s entre requests.
3. Identifica tu bot con `User-Agent` honesto (`'MyBot 1.0 (contact: email@example.com)'`).
4. **No scrapees datos personales** sin base legal (GDPR/LGPD).
5. **Lee los ToS** — algunos sitios prohíben scraping explícitamente.
6. Si hay API oficial, **úsala**. Es más fácil para ti y respeta al proveedor.

## 6️⃣ Cuándo scrapear vs cuándo NO

**Sí**:
- No hay API y los datos son públicos.
- Sitio diseñado para esto (`quotes.toscrape.com`, datasets gov.).
- Análisis personal/educativo a pequeña escala.

**No**:
- Hay API oficial → úsala.
- Datos personales sin consentimiento.
- Volumen masivo que afecta al sitio.
- ToS lo prohíbe.
- Contenido con copyright que vas a redistribuir.

## ✅ Checklist

- [ ] Parseo HTML con BeautifulSoup
- [ ] Uso find/find_all y CSS selectors
- [ ] Extraigo texto y atributos
- [ ] Sé que `requests` no ejecuta JS
- [ ] Respeto robots.txt y rate limiting
- [ ] Conozco los límites éticos/legales

## 📝 Homework

Ver `README.md`. HTML local, quotes.toscrape, robots.txt, ética.

## 📖 Definiciones y características

**Web scraping**

Extracción programática de datos de páginas HTML cuando no hay API. Pipeline típico: descargar HTML con `requests`, parsear con `BeautifulSoup`, extraer con selectores.

**BeautifulSoup**

Parser HTML tolerante a errores. `soup = BeautifulSoup(html, 'html.parser')`. Permite navegar el árbol y buscar por tag, atributo o selector CSS.

**CSS selector**

Sintaxis para localizar elementos: `'.class'`, `'#id'`, `'tag'`, `'parent > child'`, `'tag[attr=val]'`. Usados con `soup.select(...)`. Más expresivos que `find_all`.

**DOM (Document Object Model)**

Representación en árbol del HTML. Cada tag es un nodo; tiene padre, hermanos, hijos. BeautifulSoup navega este árbol con `.parent`, `.next_sibling`, `.find_all`, etc.

**`robots.txt`**

Archivo en raíz del dominio (`/robots.txt`) que declara qué paths pueden crawlear los bots. Estándar de facto; respetarlo es **legalmente** importante (variable por jurisdicción) y **éticamente** siempre.

**JS rendering**

Páginas SPA (React, Vue) cargan contenido vía JavaScript tras el HTML inicial. `requests` solo trae el HTML inicial — JS no se ejecuta. **Solución**: Playwright o Selenium (navegador headless).

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `soup.find('div')` devuelve None aunque hay divs | Buscas un atributo específico que no matchea. **Fix**: imprime `soup.prettify()[:500]` para ver el HTML real recibido; verifica clase/atributo. |
| Scrapeo y recibo HTML distinto al que veo en el navegador | El sitio renderiza con JavaScript. `requests` no ejecuta JS. **Fix**: inspecciona Network tab del navegador — quizás hay endpoint JSON que puedes consumir directo. Si no, Playwright/Selenium. |
| HTTP 403 Forbidden | El sitio detecta tu bot (User-Agent vacío o sospechoso). **Fix**: `headers={'User-Agent': 'Mozilla/5.0 ...'}` honesto, respeta robots.txt, rate limit. |
| Site funciona en navegador pero scraper devuelve `Captcha` | Anti-bot agresivo (Cloudflare, reCAPTCHA). **Fix**: respeta sus términos — si te bloquean, claramente NO quieren scraping. Busca API oficial. |
| Encoding raro (acentos `Ã¡`) | Pandas/requests dedujo encoding mal. **Fix**: `r.encoding = 'utf-8'` antes de `r.text`, o usa `r.content` (bytes) y decodifica explícito. |

## ❓ Preguntas frecuentes

**❓ ¿Scraping es legal?**

**Depende**: jurisdicción, ToS del sitio, naturaleza del dato. Datos personales = casi siempre regulado (GDPR/LGPD). Datos públicos sin ToS prohibitivo = generalmente OK. **Consulta abogado** para casos serios.

**❓ ¿`find_all` o `select`?**

**`select`** (CSS selectors) — más potente, sintaxis estándar web, más legible. **`find_all`** para casos simples o cuando integras con código heredado.

**❓ ¿Cómo descargo imágenes?**

`r = requests.get(url_imagen); open('img.jpg', 'wb').write(r.content)`. Para muchas, usa Session + thread pool.

**❓ ¿Scrapy vs BeautifulSoup?**

**BeautifulSoup**: librería de parsing. Una página, una request. **Scrapy**: framework completo (crawler, pipelines, throttling). Para proyectos serios (miles de páginas), Scrapy.

**❓ ¿Y si el sitio me bloquea?**

Respeta. Aumentar agresividad (proxies, rotating User-Agents) puede ser ilegal en algunas jurisdicciones (CFAA en USA). Considera: ¿realmente vale la pena? ¿hay otra fuente?

## 🔗 Referencias

- Mitchell, *Web Scraping with Python* 2e
- [BeautifulSoup docs](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [quotes.toscrape.com](https://quotes.toscrape.com/)

---

## 🎉 ¡Parte 0 completada!

Has cubierto los 46 fundamentos: setup, Python, NumPy, pandas, visualización, SQL/NoSQL, APIs y scraping. Tienes todo lo que el resto del programa asume.

➡️ **Siguiente:** [Parte 1 — Machine Learning clásico](../../parte-1-machine-learning-clasico/README.md) (43 clases)